<a href="https://colab.research.google.com/github/SAWGraph/public/blob/main/UseCases/UC1-Testing/UC1_CQ2_query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [ ]:
# @title Install Packages
%%capture
!pip install mapclassify
!pip install SPARQLWrapper
!pip install rdflib

In [ ]:
# @title Import packages and define functions
from branca.element import Figure                                   # For controlling the size of the final map
import folium                                                       # For map layer control
from folium.plugins import StripePattern                            # For filling polygons with a striped pattern
import geopandas as gpd                                             # For geospatial dataframes
import pandas as pd                                                 # For dataframes
from shapely import wkt                                             # For working with WKT coordinates in a GeoDataFrame
from SPARQLWrapper import SPARQLWrapper2, JSON, GET, POST, DIGEST   # For querying SPARQL endpoints
import rdflib                                                       # For working with URIs
from datetime import date                                           # For adding the date to saved map files

def convertToDataframe(results):
  d = []
  for x in results.bindings:
        row = {}
        for k in x:
            v = x[k]
            vv = rdflib.term.Literal(v.value, datatype=v.datatype).toPython()  # type: ignore[no-untyped-call]
            row[k] = vv
        d.append(row)
  df = pd.DataFrame(d)
  return df

# Question: What surface water bodies are near facilities of industry *X*?

In [ ]:
# @title Select administrative region
admin_region = "23 (Maine)" # @param [ "17 (Illinois)", "17031 (Cook County, Illinois)", "17043 (DuPage County, Illinois)", "18 (Indiana)", "18085 (Kosciusko County, Indiana)", "18097 (Marion County, Indiana)", "23 (Maine)", "23011 (Kennebec County, Maine)", "23019 (Penboscot County, Maine)", "33 (New Hampshire)" ] {"allow-input":true}

regionCode = admin_region.split()[0]
if regionCode == 'All':
  regionFilter = ''
else:
  if len(regionCode) <= 5:
    regionURI = 'kwgr:administrativeRegion.USA.' + regionCode
    print(regionURI)
    regionFilter = '''?county rdf:type kwg-ont:AdministrativeRegion_3 ;
            kwg-ont:administrativePartOf+ ''' + regionURI + ''' . '''
  else:
    # need to implement region filters with datacommons geoids.
    regionFilter = ''

kwgr:administrativeRegion.USA.23


In [ ]:
# @title Select industry
industry = '326 (Plastics and Rubber Products Manufacturing)'  #@param [ '488119 (Airports)', '327310 (Cement Manufacturing)', '325 (Chemical Manufacturing)', '3222 (Converted Paper Product Manufacturing)', '334 (Electronics Industry)', '327310 (Metal Manufacturing, Refining & Coating)', '424710 (Petroleum)', '326 (Plastics and Rubber Products Manufacturing)', '221320 (Sewage Treatment)', '562 (Waste Management)' ]
icode = industry.split()[0]
print(icode)
if len(icode) > 4:
  industryValueFilter = 'VALUES ?industryCode {naics:NAICS-' + str(icode) + '} .'
elif len(icode) == 4:
  industryValueFilter = 'VALUES ?industryGroup {naics:NAICS-' + str(icode) + '} .'
elif len(icode) == 3:
  industryValueFilter = '''?industryGroup fio:subcodeOf ?industrySubsector.
  VALUES ?industrySubsector {naics:NAICS-''' + str(icode) + '''}.'''

# extras = [ 'Consumer Products', 'Furniture and Carpet', 'Glass Products', 'Metal Ore Mining', 'Oil and Gas Extraction', 'Printing', 'Textile and Leather' ] # no facilities in Maine or runs slow
# industryLookup = {
#     'Airports': 'naics:NAICS-488119',
#     'Cement Manufacturing': 'naics:NAICS-327310',
#     'Chemical Manufacturing': 'naics:NAICS-325, naics:NAICS-424690',
#     'Consumer Products': 'naics:NAICS-442291',
#     'Converted Paper Product Manufacturing':'naics:NAICS-3222',
#     'Electronics Industry': 'naics:NAICS-334, naics:NAICS-335',
#     'Furniture and Carpet': 'naics:NAICS-561740, naics:NAICS-811420',
#     'Glass Products':'naics:NAICS-327215',
#     'Metal Manufacturing, Refining & Coating': 'naics:NAICS-327310, naics:NAICS-332812, naics:NAICS-332813, naics:NAICS-332999, naics:NAICS-333249, naics:NAICS-333316, naics:NAICS-333318, naics:NAICS-339112',
#     'Metal Ore Mining':'naics:NAICS-2122',
#     'Oil and Gas Extraction':'naics:NAICS-2111',
#     'Petroleum':'naics:NAICS-324, naics:NAICS-424710',
#     'Plastics and Rubber Products Manufacturing': 'naics:NAICS-326',
#     'Printing':'naics:NAICS-323111, naics-323120',
#     'Sewage Treatment':'naics:NAICS-221320',
#     'Textile and Leather':'naics:NAICS-313, naics:NAICS-314, naics:NAICS-316',
#     'Waste Management':'naics:NAICS-562' }
# industryCode = industryLookup[industry]

326


In [ ]:
# @title Select waterbody type
waterbody = 'hyf:HY_WaterBody' #@param ['hyf:HY_WaterBody', 'hyf:HY_FlowPath']
# Removed 'me_mgs:MGS-Well' and 'us_sdwis:PWS-ServiceArea'
print(waterbody)

hyf:HY_WaterBody


In [ ]:
# @title Choose whether to map facility S2 neighborhoods
showS2 = "True" #@param['False', 'True']
showS2 = True if showS2 == "True" else False
print(showS2)

True


# Query

In [ ]:
# @title Retrieve facilities, S2 cells, and water bodies
query = f"""
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX fio: <http://w3id.org/fio/v1/fio#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX hyf: <https://www.opengis.net/def/schema/hy_features/hyf/>
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX kwgr: <http://stko-kwg.geog.ucsb.edu/lod/resource/>
PREFIX me_mgs: <http://sawgraph.spatialai.org/v1/me_mgs#>
PREFIX naics: <http://w3id.org/fio/v1/naics#>
PREFIX nhdplusv2: <http://nhdplusv2.spatialai.org/v1/nhdplusv2#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX schema: <https://schema.org/>
PREFIX schema1: <http://schema.org/>
PREFIX spatial: <http://purl.org/spatialai/spatial/spatial-full#>

SELECT DISTINCT *
WHERE {{
    ?facility fio:ofIndustry ?industryGroup ;
              fio:ofIndustry ?industryCode ;
              kwg-ont:sfWithin ?facs2 ;
              spatial:connectedTo ?county ;
              rdfs:label ?facilityName ;
              dcterms:alternative ?facilityNameAlt ;
              schema1:address ?facilityAddress ;
			  geo:hasGeometry/geo:asWKT ?facwkt .
    {regionFilter}
    ?industryCode a naics:NAICS-IndustryCode ;
                  fio:subcodeOf ?industryGroup ;
                  rdfs:label ?industryCodeName .
    ?industryGroup rdfs:label ?industryGroupName .
    {industryValueFilter}
    ?facs2 rdf:type kwg-ont:S2Cell_Level13 ;
           kwg-ont:sfTouches|owl:sameAs ?nbrs2 ;
           geo:hasGeometry/geo:asWKT ?facs2wkt .
	?nbrs2 spatial:connectedTo ?waterbody ;
           geo:hasGeometry/geo:asWKT ?nbrs2wkt .
    ?waterbody rdf:type {waterbody} ;
	           nhdplusv2:hasFTYPE ?wbftype ;
		       nhdplusv2:hasReachCode ?wbreachcode ;
               geo:hasGeometry/geo:asWKT ?wbwkt .
    OPTIONAL{{ ?waterbody  schema:name ?wbname }}
}}
"""

print(query)


PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX fio: <http://w3id.org/fio/v1/fio#>
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX hyf: <https://www.opengis.net/def/schema/hy_features/hyf/>
PREFIX kwg-ont: <http://stko-kwg.geog.ucsb.edu/lod/ontology/>
PREFIX kwgr: <http://stko-kwg.geog.ucsb.edu/lod/resource/>
PREFIX me_mgs: <http://sawgraph.spatialai.org/v1/me_mgs#>
PREFIX naics: <http://w3id.org/fio/v1/naics#>
PREFIX nhdplusv2: <http://nhdplusv2.spatialai.org/v1/nhdplusv2#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX schema: <https://schema.org/>
PREFIX schema1: <http://schema.org/>
PREFIX spatial: <http://purl.org/spatialai/spatial/spatial-full#>

SELECT DISTINCT *
WHERE {
    ?facility fio:ofIndustry ?industryGroup ;
              fio:ofIndustry ?industryCode ;
              kwg-ont:sfWithin ?facs2 ;
              spatial:connectedTo ?county ;
       

In [ ]:
%%time
endpointGET = 'https://apps.okn.us/federation/sparql'

sparqlGET = SPARQLWrapper2(endpointGET)
sparqlGET.setHTTPAuth(DIGEST)
sparqlGET.setMethod(POST)
sparqlGET.setReturnFormat(JSON)

sparqlGET.setQuery(query)
query_result = sparqlGET.query()
result = convertToDataframe(query_result)
print(result.tail())
if result.empty:
    raise Exception("No results returned for chosen input parameters.")

                                              facility  \
185  http://w3id.org/fio/v1/epa-frs-data#d.FRS-Faci...   
186  http://w3id.org/fio/v1/epa-frs-data#d.FRS-Faci...   
187  http://w3id.org/fio/v1/epa-frs-data#d.FRS-Faci...   
188  http://w3id.org/fio/v1/epa-frs-data#d.FRS-Faci...   
189  http://w3id.org/fio/v1/epa-frs-data#d.FRS-Faci...   

                                industryGroup  \
185   http://w3id.org/fio/v1/naics#NAICS-3262   
186  http://w3id.org/fio/v1/naics#NAICS-32621   
187  http://w3id.org/fio/v1/naics#NAICS-32621   
188  http://w3id.org/fio/v1/naics#NAICS-32621   
189  http://w3id.org/fio/v1/naics#NAICS-32621   

                                  industryCode  \
185  http://w3id.org/fio/v1/naics#NAICS-326211   
186  http://w3id.org/fio/v1/naics#NAICS-326211   
187  http://w3id.org/fio/v1/naics#NAICS-326211   
188  http://w3id.org/fio/v1/naics#NAICS-326211   
189  http://w3id.org/fio/v1/naics#NAICS-326211   

                                                 facs2 

# Prep data for mapping

In [ ]:
# @title Create thematic dataframes
facilities = result[['facility', 'facilityName', 'facilityNameAlt', 'facilityAddress', 'industryCode', 'industryCodeName', 'industryGroup', 'industryGroupName', 'facwkt']].copy()
facilities['facility'] = facilities['facility'].apply(lambda x: f'<a href="https://frs-public.epa.gov/ords/frs_public2/fii_query_detail.disp_program_facility?p_registry_id={x.split(".")[-1]}" target="_blank">{x}</a>')
facilities.drop_duplicates(inplace=True)
facilities['facwkt'] = facilities['facwkt'].apply(wkt.loads)

if showS2:
    facs2 = result[['facs2', 'facs2wkt']].copy()
    facs2.rename(columns={'facs2': 'nbrs2', 'facs2wkt': 'nbrs2wkt'}, inplace=True)
    nbrs2 = result[['nbrs2', 'nbrs2wkt']].copy()
    nbrs2 = pd.concat([facs2, nbrs2], ignore_index=True)
    nbrs2['nbrs2'] = nbrs2['nbrs2'].apply(lambda x: f'<a href="{x}" target="_blank">{x}</a>')
    nbrs2.drop_duplicates(inplace=True)
    nbrs2['nbrs2wkt'] = nbrs2['nbrs2wkt'].apply(wkt.loads)

wb_attrs = []
for attr in ['wbname', 'wbftype', 'wbreachcode', 'waterbody', 'wbwkt']:
    if attr in result.columns:
        wb_attrs.append(attr)
waterbodies = result[wb_attrs].copy()
waterbodies.drop_duplicates(inplace=True)
waterbodies['wbwkt'] = waterbodies['wbwkt'].apply(wkt.loads)

In [ ]:
# @title Create GeoDataFrames
facilities = gpd.GeoDataFrame(facilities, geometry='facwkt')
facilities.set_crs(epsg=4326, inplace=True, allow_override=True)

if showS2:
    nbrs2 = gpd.GeoDataFrame(nbrs2, geometry='nbrs2wkt')
    nbrs2.set_crs(epsg=4326, inplace=True, allow_override=True)

waterbodies = gpd.GeoDataFrame(waterbodies, geometry='wbwkt')
waterbodies.set_crs(epsg=4326, inplace=True, allow_override=True)

,wbname,wbftype,wbreachcode,waterbody,wbwkt
0,NaN,Reservoir,None,https://geoconnex.us/nhdplusv2/comid/120050455,"POLYGON ((-70.14588 44.08272, -70.146 44.08252..."
12,NaN,SwampMarsh,None,https://geoconnex.us/nhdplusv2/comid/6710983,"POLYGON ((-69.72969 44.35554, -69.73014 44.355..."
36,Adams Pond,LakePond,01050003001332,https://geoconnex.us/nhdplusv2/comid/5204596,"POLYGON ((-69.63047 43.8971, -69.63037 43.8963..."
37,NaN,LakePond,01050003001333,https://geoconnex.us/nhdplusv2/comid/5204602,"POLYGON ((-69.63758 43.88798, -69.63761 43.888..."
40,NaN,LakePond,01050003001331,https://geoconnex.us/nhdplusv2/comid/5204598,"POLYGON ((-69.61839 43.89347, -69.61731 43.893..."
41,Wiley Pond,LakePond,01050003001329,https://geoconnex.us/nhdplusv2/comid/5204590,"POLYGON ((-69.63714 43.89713, -69.63739 43.897..."
43,NaN,Reservoir,None,https://geoconnex.us/nhdplusv2/comid/120050504,"POLYGON ((-70.71694 43.41275, -70.71665 43.412..."
44,NaN,Reservoir,None,https://geoconnex.us/nhdplusv2/comid/120050505,"POLYGON ((-70.72399 43.41238, -70.72493 43.412..."
55,NaN,SwampMarsh,None,https://geoconnex.us/nhdplusv2/comid/19334793,"POLYGON ((-70.13452 44.20968, -70.13455 44.209..."
67,NaN,LakePond,01060001001360,https://geoconnex.us/nhdplusv2/comid/6720129,"POLYGON ((-70.40315 43.64426, -70.40482 43.645..."


In [ ]:
# @title Create map layers
%%capture
fac_color = 'red'
s2_color = 'dimgray'
wb_color = 'blue'
boundweight = 5

map = folium.Map(tiles=None)
folium.TileLayer('openstreetmap', name='OpenStreetMap', show=True, referrerPolicy='strict-origin-when-cross-origin').add_to(map)
# folium.TileLayer('CartoDB voyager', name='Carto Voyager', show=False).add_to(map)

sp = StripePattern(angle=-30, color=s2_color, space_color='white', space_opacity=0.75).add_to(map)

if showS2:
    nbrs2.explore(m=map,
                  color=s2_color,
                  style_kwds=dict(weight=0.5*boundweight, style_function=lambda x :{'fillPattern': sp}),
                  popup=['nbrs2'],
                  tooltip=False,
                  name=f'<span style="color: {s2_color};">Facility S2 Cells</span>',
                  show=False)
facilities.explore(m=map,
                   color=fac_color,
                   style_kwds=dict(weight=2*boundweight),
                   popup=['facilityName', 'facilityNameAlt', 'facilityAddress', 'industryGroupName', 'industryCodeName', 'facility'],
                   tooltip=False,
                   name=f'<span style="color: {fac_color};">{industry} Facilities</span>',
                   show=True)
waterbodies.explore(m=map,
                    color=wb_color,
                    style_kwds=dict(weight=boundweight),
                    popup=wb_attrs[:-1],
                    tooltip=False,
                    name=f'<span style="color: {wb_color};">Water Bodies</span>',
                    show=True)

# Map

In [ ]:
# @title Display map
print()
print(f"Instances of {waterbody} near NAICS {industry} facility locations within admin region {admin_region}.")
print()
print('Use the checkboxes on the right to turn water bodies or facilities layers on/off.')
print('Left-click and drag to pan the map. Use the mouse wheel or +/- buttons on the left to zoom in/out.')
print()
folium.FitOverlays().add_to(map)
folium.LayerControl(collapsed=False).add_to(map)
fig = Figure(width='90%', height='700')
fig.add_child(map)


Instances of hyf:HY_WaterBody near NAICS 326 (Plastics and Rubber Products Manufacturing) facility locations within admin region 23 (Maine).

Use the checkboxes on the right to turn water bodies or facilities layers on/off.
Left-click and drag to pan the map. Use the mouse wheel or +/- buttons on the left to zoom in/out.



In [ ]:
# @title Save .html copy of map
today = date.today()
fig.save(f'UC1-CQ2_{waterbody}_near-{"".join(industry.split())}_in-{"".join(admin_region.split())}_{today}.html')